# [개별미션 4차] 헬스케어 기술보증 데이터 분석

## 0. 준비

In [ ]:
# 코랩에 없는 라이브러리 설치
!pip install konlpy gensim

In [ ]:
# 데이터 파일 업로드 (06_Tech_Credit_Data.xlsx 선택)
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('06_Tech_Credit_Data.xlsx', sheet_name='기술데이터')
print(df.shape)
df.head()

In [ ]:
# 데이터 정보 확인
df.info()

In [ ]:
# 목표 변수: 진행상태 (여신승인 / 여신거부)
print(df['진행상태'].value_counts())

In [ ]:
# 형태소 분석기 불러오기
from konlpy.tag import Okt
okt = Okt()

# 예시 문장으로 명사 추출 확인
print(okt.nouns('실시간 대시보드를 적용한 감염관리 모니터링 시스템 개발'))

---
## 1. 대출 승인 기업 중 자본금 상위 100개 기업의 기술명 최빈 단어

In [ ]:
# 자본금은 이미 숫자형(int64)이라 변환 없이 사용
print(df['자본금'].dtype)

# 대출 승인 기업 중 자본금 상위 100개 추출
top100_approved = (
    df[df['진행상태'] == '여신승인']
    .sort_values(by='자본금', ascending=False)
    .head(100)
)
top100_approved[['회사명', '기술명', '자본금']].head()

In [ ]:
# Okt로 명사 추출 (한 글자 단어는 의미가 적어서 2글자 이상만)
nouns_list = []
for text in top100_approved['기술명']:
    nouns = okt.nouns(text)
    nouns_list.extend([n for n in nouns if len(n) > 1])

# 단어 빈도수 (동점이 많아서 15개까지 확인)
word_counts = pd.Series(nouns_list).value_counts()
print('=== 전체 명사 빈도 상위 15 ===')
display(word_counts.head(15))

In [ ]:
# '개발, 기술, 제조' 같은 단어는 어느 기술명에나 붙어서 기술 내용을 알려주지 않음
# → 불용어로 빼야 하는데, 눈으로 고르지 않고 데이터로 찾기
#    기준: 주요 세부분류 20개 중 몇 개 분야에 등장하는 단어인지 세기
#          → 거의 모든 분야에 나오는 단어 = 분야와 상관없는 범용어

top20_fields = df['헬스케어 세부분류'].value_counts().head(20).index

word_field_count = {}
for field in top20_fields:
    field_words = set()            # 한 분야 안에서는 한 번만 세기
    for text in df[df['헬스케어 세부분류'] == field]['기술명']:
        field_words.update([n for n in okt.nouns(text) if len(n) > 1])
    for w in field_words:
        word_field_count[w] = word_field_count.get(w, 0) + 1

field_spread = pd.Series(word_field_count).sort_values(ascending=False)
field_spread.head(20)

In [ ]:
# 16~19개 분야에 나오는 단어와 9개 이하 단어 사이가 크게 벌어짐 → 15개 이상을 범용어로 정함
stopwords = list(field_spread[field_spread >= 15].index)

# '~을 이용한', '~기반' 처럼 문장을 잇는 표현에서 나온 단어도 추가
stopwords = stopwords + ['이용', '기반']
print('불용어:', stopwords)

word_counts_sw = pd.Series([n for n in nouns_list if n not in stopwords]).value_counts()
print('=== 불용어 제외 상위 15 ===')
display(word_counts_sw.head(15))

In [ ]:
# (참고) 수업에서 배운 BoW(CountVectorizer)로 세어도 같은 결과
from sklearn.feature_extraction.text import CountVectorizer

texts = [' '.join([n for n in okt.nouns(t) if len(n) > 1]) for t in top100_approved['기술명']]
cv = CountVectorizer(stop_words=stopwords)
bow = cv.fit_transform(texts)

bow_counts = pd.Series(bow.toarray().sum(axis=0), index=cv.get_feature_names_out())
bow_counts.sort_values(ascending=False).head(10)

### 1번 결과
- 전체 명사 기준으로는 **'개발'(36회)**이 가장 많고, 그다음이 '기술'(30회)이다.
- 하지만 '개발·기술·제조' 같은 단어는 주요 세부분류 20개 중 16~19개 분야에 모두 나오는 범용어라, 기술 내용을 알려주지 않는다. 이런 단어를 빼고 다시 세면 **'기능'(21) · '건강'(20) · '치료'(19)** · '식품'(13) · '소재'(13) · '진단'(12) 순이다.
- '건강·기능·식품'은 '건강기능식품'이 나뉘어 나온 단어다.
- **→ 자본금이 큰 승인 기업은 주로 건강기능식품, 치료, 진단 관련 기술을 가지고 있다.**

---
## 2. 업력 계산 및 대출 승인 여부에 따른 업력 차이 검정

In [ ]:
# 설립연도가 1900인 데이터는 실제 연도가 아니라 '모름'을 뜻하는 값 → 제외
print('설립연도 1900 개수:', (df['설립연도'] == 1900).sum())
df2 = df[df['설립연도'] != 1900].copy()

# 업력 = 현재 연도 - 설립연도
from datetime import datetime
this_year = datetime.now().year
df2['업력'] = this_year - df2['설립연도']

df2.groupby('진행상태')['업력'].describe()

In [ ]:
from scipy import stats

approve = df2[df2['진행상태'] == '여신승인']['업력']
reject = df2[df2['진행상태'] == '여신거부']['업력']

# 1) 정규성 검정 (p < 0.05 이면 정규분포가 아님)
print('정규성 - 승인:', stats.shapiro(approve).pvalue)
print('정규성 - 거부:', stats.shapiro(reject).pvalue)

In [ ]:
# 2) 정규분포가 아니므로 비모수 검정인 Mann-Whitney U 검정 (중앙값 비교)
#    귀무가설: 승인 기업과 거부 기업의 업력에 차이가 없다
print('승인 중앙값:', approve.median(), '/ 거부 중앙값:', reject.median())
print(stats.mannwhitneyu(approve, reject))

# (참고) 평균 비교 t-검정
print('승인 평균:', round(approve.mean(), 2), '/ 거부 평균:', round(reject.mean(), 2))
print(stats.ttest_ind(approve, reject))

### 2번 결과
- 업력 중앙값: 승인 **19년**, 거부 **19년** (평균 20.47년 vs 20.49년)
- 두 그룹 모두 정규성 검정 p < 0.05 → 정규분포가 아니어서 Mann-Whitney U 검정을 사용했다.
- Mann-Whitney U 검정 **p = 0.84** (t-검정도 p = 0.93) → p > 0.05이므로 **귀무가설을 기각하지 못한다.**
- **→ 대출 승인 여부에 따라 기업 업력에 유의미한 차이는 없다.**

---
## 3. 평가등급과 대출 승인 여부 가설 검정 + 등급별 승인 비율

In [ ]:
# 평가등급 × 진행상태 교차표
grade_table = pd.crosstab(df['평가등급'], df['진행상태'])

# 등급별 승인 비율
grade_table['합계'] = grade_table['여신거부'] + grade_table['여신승인']
grade_table['승인비율'] = (grade_table['여신승인'] / grade_table['합계']).round(3)
grade_table.sort_values('합계', ascending=False)

In [ ]:
# 카이제곱 검정은 칸마다 개수가 충분해야 함 (기대빈도 5 이상)
# 'T3/A', 'AAA', 'CCC'처럼 10건도 안 되는 등급이 많아서, 개수가 많은 5개 등급만 사용
main_grades = ['AA', 'A', 'BBB', 'BB', 'B']
df3 = df[df['평가등급'].isin(main_grades)]
print('검정에 사용한 데이터:', len(df3), '/ 전체:', len(df))

table = pd.crosstab(df3['평가등급'], df3['진행상태'])
table = table.loc[main_grades]      # 등급 순서대로 정렬
table['승인비율'] = (table['여신승인'] / (table['여신거부'] + table['여신승인'])).round(3)
table

In [ ]:
# 카이제곱 독립성 검정
# 귀무가설: 평가등급과 대출 승인 여부는 관계가 없다(독립이다)
chi2, p, dof, expected = stats.chi2_contingency(table[['여신거부', '여신승인']])
print('카이제곱 통계량:', round(chi2, 3))
print('p-value:', round(p, 4))
print('자유도:', dof)

### 3번 결과
| 평가등급 | AA | A | BBB | BB | B |
|---|---|---|---|---|---|
| 승인비율 | 87.4% | 81.7% | 83.5% | 81.5% | 79.5% |

- 등급이 낮을수록 승인비율이 조금 낮아지지만 차이는 8%p 이내다.
- 카이제곱 검정 **p = 0.289** → p > 0.05이므로 **귀무가설을 기각하지 못한다.**
- AAA(2건), CCC(2건) 같은 등급의 100%나 0%는 건수가 너무 적어서 의미가 없다.
- **→ 평가등급과 대출 승인 여부 사이에 유의미한 관계는 없다.**

---
## 4. '병원/의료 IT' 기술명 단어 표현(Word2Vec) → '감염'과 유사한 단어

In [ ]:
# 병원/의료 IT 기업만 추출
it_df = df[df['헬스케어 세부분류'] == '병원/의료 IT']
print('기업 수:', len(it_df))

# 기술명마다 명사 리스트 만들기 (Word2Vec 입력 형태: [['단어', '단어'], ['단어', ...], ...])
sentences = []
for text in it_df['기술명']:
    nouns = okt.nouns(text)
    sentences.append([n for n in nouns if len(n) > 1])

# 'EMR', 'Smart Hospital'처럼 영어로만 된 기술명은 Okt가 명사를 뽑지 못해 빈 리스트가 됨
print(sentences[:5])
print('빈 리스트 개수:', sum(1 for s in sentences if len(s) == 0))

In [ ]:
# '감염'이 들어간 기술명 확인
print('감염 등장 횟수:', sum(s.count('감염') for s in sentences))
it_df[it_df['기술명'].str.contains('감염')]['기술명']

In [ ]:
from gensim.models import Word2Vec

# Skip-Gram (sg=1): 중심 단어로 주변 단어를 예측 → 데이터가 적고 드문 단어에 유리
# 문장이 183개로 적어서 min_count=1(모든 단어 사용), epochs를 크게
model_sg = Word2Vec(sentences, vector_size=100, window=3, min_count=1, sg=1, epochs=200, seed=42, workers=1)

print("=== Skip-Gram: '감염'과 유사한 단어 ===")
for word, score in model_sg.wv.most_similar('감염', topn=10):
    print(word, round(score, 3))

In [ ]:
# CBOW (sg=0): 주변 단어로 중심 단어를 예측
model_cbow = Word2Vec(sentences, vector_size=100, window=3, min_count=1, sg=0, epochs=200, seed=42, workers=1)

print("=== CBOW: '감염'과 유사한 단어 ===")
for word, score in model_cbow.wv.most_similar('감염', topn=10):
    print(word, round(score, 3))

In [ ]:
# 유사어로 나온 단어가 어떤 기술명에 쓰였는지 확인
for word in ['수술실', '병상']:
    print(f'[{word}]')
    print(it_df[it_df['기술명'].str.contains(word)]['기술명'].head(3).tolist())

### 4번 결과
- '감염'과 유사한 단어 (Skip-Gram): **모니터링, 스마트, 솔루션, 수술실, 병상, 스케줄링**
- '감염'은 9번 나오는데, 모두 `감염관리 모니터링 시스템`이라는 표현이다. 그래서 항상 같이 나오는 '모니터링'이 가장 가깝게 나왔다.
- '수술실', '병상'은 `수술실 스케줄링 시스템`, `스마트 병상 관리 시스템`처럼 '감염'과 같은 자리에 쓰이는 단어다. 즉 **병원에서 관리하는 대상**이라는 점에서 비슷하다.
- CBOW는 '관리, 적용, 시스템'처럼 자주 나오는 단어가 많이 섞였다. 데이터가 적을 때는 Skip-Gram 결과가 더 의미 있다.
- 한계: 문장이 183개로 적어서 실행할 때마다 순위가 조금 바뀔 수 있다.

---
## 5. 대출금액을 가장 많이 집행한 은행 → 해당 은행의 세부분류별 대출금액

In [ ]:
# 대출금액에 '금액없음' 문자가 섞여 있음 → 숫자형인 '대출금' 컬럼 사용 (금액없음 = 0)
print(df['대출금액'].unique()[:10])

# 은행별 대출금 합계 (실제 집행된 = 승인된 건만)
approved_df = df[df['진행상태'] == '여신승인']
bank_sum = approved_df.groupby('신청은행')['대출금'].sum().sort_values(ascending=False)
bank_sum

In [ ]:
top_bank = bank_sum.index[0]
print('대출금액을 가장 많이 집행한 은행:', top_bank)
print('전체 대비 비중:', round(bank_sum.iloc[0] / bank_sum.sum() * 100, 1), '%')

In [ ]:
# 해당 은행의 헬스케어 세부분류별 대출금 합계
bank_df = approved_df[approved_df['신청은행'] == top_bank]
bank_sub = bank_df.groupby('헬스케어 세부분류')['대출금'].agg(['sum', 'count']).sort_values('sum', ascending=False)
bank_sub.head(10)

### 5번 결과
- 대출금액을 가장 많이 집행한 은행은 **기업은행**이다. 84,285로 전체의 **51.6%**를 차지한다.
- 기업은행이 가장 많이 집행한 세부분류는 **의료영상 AI**(7,704, 117건)다.
- 다만 2위 건강기능식품(7,370), 3위 의료영상 장치(7,310)와 차이가 크지 않아, 한 분야에 몰려 있지는 않다.

---
## 6. 기술명·대출금액·기술분류·신청은행·자본금 → 대출 승인 여부 예측 (Scikit-learn)
- 기술명(텍스트) → **TF-IDF**
- 기술분류, 신청은행(범주) → **원-핫 인코딩**(get_dummies)
- 대출금액, 자본금(숫자) → 그대로 사용

In [ ]:
# 목표 변수: 승인 = 1, 거부 = 0
df['승인'] = (df['진행상태'] == '여신승인').astype(int)

# 대출금액의 '금액없음'을 0으로 바꾸고 숫자로 변환
df['대출금액_숫자'] = pd.to_numeric(df['대출금액'], errors='coerce').fillna(0)

# 기술명을 명사만 남긴 문장으로 변환 (TF-IDF 입력용)
df['기술명_명사'] = [' '.join([n for n in okt.nouns(t) if len(n) > 1]) for t in df['기술명']]
df[['기술명', '기술명_명사']].head()

In [ ]:
# 범주형 → 원-핫 인코딩, 숫자형은 그대로
etc = pd.get_dummies(df[['기술분류', '신청은행']]).astype(int)
etc['대출금액'] = df['대출금액_숫자']
etc['자본금'] = df['자본금']
print(etc.shape)
etc.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 학습용 80% / 평가용 20% 나누기 (stratify: 승인/거부 비율을 똑같이 유지)
train_idx, test_idx = train_test_split(df.index, test_size=0.2, stratify=df['승인'], random_state=42)

# TF-IDF는 학습 데이터로만 fit (평가 데이터 정보가 섞이지 않도록)
tfidf = TfidfVectorizer(max_features=300)
train_text = tfidf.fit_transform(df.loc[train_idx, '기술명_명사']).toarray()
test_text = tfidf.transform(df.loc[test_idx, '기술명_명사']).toarray()

# 텍스트 특성 + 나머지 특성 합치기
X_train = np.hstack([train_text, etc.loc[train_idx].values])
X_test = np.hstack([test_text, etc.loc[test_idx].values])
y_train = df.loc[train_idx, '승인']
y_test = df.loc[test_idx, '승인']
print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 랜덤포레스트 (거부가 적어서 class_weight='balanced'로 균형 맞춤)
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print('정확도:', round(accuracy_score(y_test, pred), 3))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=['여신거부', '여신승인']))

In [ ]:
# 어떤 변수가 중요했는지 확인
feature_names = list(tfidf.get_feature_names_out()) + list(etc.columns)
importance = pd.Series(rf.feature_importances_, index=feature_names)
importance.sort_values(ascending=False).head(10)

### 성능이 너무 높다 → 대출금액 확인
대출금액이 가장 중요한 변수로 나왔다. 대출금액과 승인 여부를 직접 비교해본다.

In [ ]:
# 대출금액이 0인지 여부 × 진행상태
pd.crosstab(df['대출금액_숫자'] == 0, df['진행상태'])

In [ ]:
# 대출금액을 빼고 다시 학습 (기술명, 기술분류, 신청은행, 자본금만 사용)
X_train2 = np.hstack([train_text, etc.drop(columns='대출금액').loc[train_idx].values])
X_test2 = np.hstack([test_text, etc.drop(columns='대출금액').loc[test_idx].values])

rf2 = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf2.fit(X_train2, y_train)
pred2 = rf2.predict(X_test2)

print('정확도:', round(accuracy_score(y_test, pred2), 3))
print(confusion_matrix(y_test, pred2))
print(classification_report(y_test, pred2, target_names=['여신거부', '여신승인']))

### 6번 결과
- 5개 변수를 모두 넣은 랜덤포레스트: **정확도 98.6%**
- 그런데 변수 중요도에서 **대출금액이 0.72**로, 2위 자본금(0.02)과 비교할 수 없을 만큼 높다.
- 교차표를 확인하니 **거부 405건 중 404건이 대출금액 0**이었다. 대출금액은 승인된 뒤에 정해지는 금액이라, 사실상 답을 알려주는 변수다.
- 대출금액을 빼고 다시 학습하면 정확도가 74%로 떨어지고, 거부를 거의 맞히지 못한다(거부 81건 중 10건). 전부 승인으로 찍어도 81.7%라서, 대출금액 없이는 예측이 안 된다고 봐야 한다.
- **→ 모델은 만들었지만, 높은 성능은 대출금액 덕분이다. 실제 심사 전에 쓰려면 대출금액 대신 신청금액 같은 정보가 필요하다.**

---
## 7. 기술명 → 대출 승인 여부 신경망 모델 (model_loan.h5 저장)
순서: 텍스트 → 정수 시퀀스(Text to Sequence) → 길이 맞추기(Padding) → Embedding + LSTM

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 6번에서 나눈 학습/평가 데이터 그대로 사용
train_texts = df.loc[train_idx, '기술명_명사']
test_texts = df.loc[test_idx, '기술명_명사']

# 1) Tokenizer: 단어마다 번호 붙이기 (학습 데이터 기준, 처음 보는 단어는 'OOV')
tokenizer = Tokenizer(oov_token='OOV')
tokenizer.fit_on_texts(train_texts)
vocab_size = len(tokenizer.word_index) + 1     # +1은 패딩(0)용
print('단어 수:', vocab_size)

# 2) Text to Sequence: 문장 → 숫자 리스트
train_seq = tokenizer.texts_to_sequences(train_texts)
test_seq = tokenizer.texts_to_sequences(test_texts)
print(train_texts.iloc[0], '→', train_seq[0])

In [ ]:
# 3) Padding: 길이를 똑같이 맞추기 (가장 긴 문장 길이 기준)
max_len = max(len(s) for s in train_seq)
print('최대 길이:', max_len)

X_train_pad = pad_sequences(train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(test_seq, maxlen=max_len, padding='post')
print(X_train_pad[0])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import tensorflow as tf

tf.random.set_seed(42)

model = Sequential()
model.add(Embedding(vocab_size, 64))            # 단어 번호 → 64차원 벡터
model.add(LSTM(32))                             # 단어 순서를 고려해 문장 특징 추출
model.add(Dropout(0.5))                         # 과적합 방지
model.add(Dense(1, activation='sigmoid'))       # 승인 확률 (0~1)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# 거부 데이터가 적어서 거부에 가중치를 더 줌 (거부 405건 : 승인 1808건 ≈ 1 : 4.5)
class_weight = {0: 4.5, 1: 1}

history = model.fit(X_train_pad, y_train, epochs=10, batch_size=32,
                    validation_split=0.2, class_weight=class_weight)

In [ ]:
# 평가
loss, acc = model.evaluate(X_test_pad, y_test)
print('평가 정확도:', round(acc, 3))
print('(비교) 전부 승인으로 찍었을 때 정확도:', round(y_test.mean(), 3))

pred_nn = (model.predict(X_test_pad) > 0.5).astype(int).ravel()
print(confusion_matrix(y_test, pred_nn))
print(classification_report(y_test, pred_nn, target_names=['여신거부', '여신승인']))

In [ ]:
# 학습 과정 그래프 (학습 정확도 vs 검증 정확도)
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.show()

In [ ]:
# 모델 저장
model.save('model_loan.h5')

# 저장한 모델 불러와서 새 기술명 예측해보기
from tensorflow.keras.models import load_model
loaded_model = load_model('model_loan.h5')

new_text = 'AI 기반 감염관리 모니터링 시스템 개발'
new_seq = tokenizer.texts_to_sequences([' '.join([n for n in okt.nouns(new_text) if len(n) > 1])])
new_pad = pad_sequences(new_seq, maxlen=max_len, padding='post')
print(new_text, '→ 승인 확률:', round(float(loaded_model.predict(new_pad)[0][0]), 3))

In [ ]:
# 코랩에서 파일 다운로드
files.download('model_loan.h5')

### 7번 결과
- 기술명 → Tokenizer(단어 1,019개) → 시퀀스 변환 → 패딩(길이 16) → Embedding + LSTM 모델을 만들고 **model_loan.h5로 저장**했다.
- 평가 정확도는 **약 45%**다. 전부 승인으로 찍어도 81.7%이므로 훨씬 낮다.
- 그래프를 보면 학습 정확도는 계속 오르는데 검증 정확도는 오르지 않는다. 기술명을 외울 뿐 새로운 데이터에는 맞지 않는다는 뜻이다.
- **→ 기술명만으로는 대출 승인 여부를 예측하기 어렵다.** 6번에서 대출금액을 뺐을 때 성능이 떨어진 것과 같은 결론이다.
- (신경망은 실행할 때마다 결과가 조금씩 다를 수 있다.)

---
## 종합 정리
- **업력(2번), 평가등급(3번), 기술명(6·7번)** 모두 대출 승인 여부와 뚜렷한 관계가 없었다.
- 승인을 잘 맞히는 변수는 **대출금액**뿐인데, 이는 승인 결과로 정해지는 값이라 사전 예측에는 쓸 수 없다.
- 자금은 **기업은행**에 절반 이상(51.6%) 몰려 있고, 기업은행 안에서는 의료영상 AI·건강기능식품·의료영상 장치에 고르게 나뉘어 있다.
- **제안**: 승인을 미리 예측하려면 신청금액, 매출·부채비율, 특허 수, 인허가 단계처럼 **심사 전에 알 수 있는 정보**를 더 모아야 한다. 또 평가등급이 승인과 거의 관계없다는 점은, 기술평가 결과가 실제 심사에 잘 반영되고 있는지 점검이 필요하다는 뜻이다.